# Multi-task Computer Vision on PASCAL VOC2012

**Portfolio walkthrough — Tanjim Hossain**

This notebook is a compact, recruiter-facing companion to the full academic experiment. It documents the locked protocol, headline results, and reusable selected architectures for three tasks:

1. multi-label image classification,
2. binary foreground/background semantic segmentation,
3. 20-class object detection.

The original GPU run used the official VOC2012 train partitions for development and retained the official VOC2012 validation partitions as one-time final holdouts.


## 1. Experiment protocol

Model selection used **training + internal validation only**. The final holdout was not used for architecture comparison, epoch selection, threshold tuning, or NMS tuning.


In [ ]:
import pandas as pd

splits = pd.DataFrame({
    "task": ["classification", "segmentation", "detection"],
    "training": [4574, 1171, 4574],
    "internal_validation": [1143, 293, 1143],
    "final_holdout": [5823, 1449, 5823],
})
splits


## 2. Locked final results

These values come from the preserved final academic run; they are not recomputed in this lightweight walkthrough.


In [ ]:
results = pd.DataFrame([
    {"task": "Multi-label classification", "selected_model": "Frozen Xception", "primary_metric": "PR-AUC", "value": 0.81444},
    {"task": "Binary segmentation", "selected_model": "U-Net", "primary_metric": "Foreground IoU", "value": 0.51007},
    {"task": "Object detection", "selected_model": "Xception-backed YOLO-style", "primary_metric": "mAP@0.50", "value": 0.19000},
])
results


## 3. Selected classification architecture

The final classifier uses an ImageNet-pretrained Xception backbone in frozen inference mode plus the selected batch-normalised bottleneck head. The repository model builder reproduces the architecture and preprocessing choices from the locked experiment.


In [ ]:
from voc_multitask.models import build_frozen_xception_classifier

# Requires the full TensorFlow/Keras dependencies from requirements.txt.
# model = build_frozen_xception_classifier()
# model.summary()


### Why this model was selected

The regularised scratch CNN improved substantially over the deliberately overfit baseline, but frozen Xception transfer learning was much stronger. Restricted last-block fine-tuning increased trainable capacity without improving the predeclared validation BCE criterion, so the simpler frozen checkpoint was retained.


In [ ]:
classification_comparison = pd.DataFrame([
    ["High-capacity baseline", 16_258_900, 0.22095, 0.26261],
    ["Regularised efficient CNN", 681_108, 0.18454, 0.40043],
    ["Extended frozen Xception", 542_548, 0.10021, 0.80487],
    ["Last-block fine-tuned Xception", 5_284_180, 0.10021, 0.80492],
], columns=["model", "trainable_or_total_params", "val_BCE", "val_PR_AUC"])
classification_comparison


## 4. Selected segmentation architecture

The U-Net is parameter-matched to the encoder-decoder baseline. Same-scale skip connections recover spatial detail lost during downsampling and improved internal-validation foreground IoU from **0.46599** to **0.51003**.


In [ ]:
from voc_multitask.models import build_segmentation_unet

# unet = build_segmentation_unet()
# unet.summary()


## 5. Selected detector architecture

The detector uses a frozen ImageNet Xception backbone at 8×8 spatial resolution, two box slots per grid cell, a separable-convolution prediction head, VOC difficult-object handling, and class-aware NMS. It was selected by internal-validation mAP@0.50 rather than the composite training loss.


In [ ]:
from voc_multitask.models import build_yolo_style_detector

# detector = build_yolo_style_detector()
# detector.output_shape
# Expected: (None, 8, 8, 2, 25)


## 6. Metric sanity checks

Reusable overlap and box-IoU helpers live in `src/voc_multitask/metrics.py` and are covered by lightweight unit tests.


In [ ]:
import numpy as np
from voc_multitask.metrics import binary_iou, dice_score, box_iou_one_to_many

y_true = np.array([[1, 1], [0, 0]])
y_prob = np.array([[0.9, 0.8], [0.7, 0.1]])
print("IoU:", binary_iou(y_true, y_prob))
print("Dice:", dice_score(y_true, y_prob))

box = np.array([0.0, 0.0, 1.0, 1.0])
boxes = np.array([[0.0, 0.0, 1.0, 1.0], [0.5, 0.5, 1.5, 1.5]])
print("Box IoU:", box_iou_one_to_many(box, boxes))


## 7. Reproduction notes

For the full environment:

```bash
pip install -r requirements.txt
```

Place PASCAL VOC2012 under `VOCdevkit/VOC2012/` or set `VOC_ROOT` to the extracted dataset directory. Full retraining is GPU-intensive; the original experiments ran on Kaggle using two NVIDIA Tesla T4 GPUs with Python 3.12.13, TensorFlow 2.20.0, and Keras 3.13.2.

For methodology, model comparisons, failure analysis, and all locked metrics, see [`../docs/technical_report.md`](../docs/technical_report.md).
